# 🌍 Global Health & Life Expectancy — Advanced Data Visualization
### ITS68404 · Data Visualization · Group Assignment
**SDG 3 — Good Health and Well-Being**

| | |
|---|---|
| **Dataset** | WHO Life Expectancy (2938 rows × 22 cols) + Gapminder (3675 rows × 8 cols) |
| **Merged** | 2,416 records · 151 countries · 6 continents · 2000–2015 |
| **Tools** | Python · Pandas · NumPy · Scikit-learn · **Plotly** |

---

## 📋 Task 1 — Problem Identification & Research Context

## Real-World Problem
**How do socioeconomic and healthcare factors influence life expectancy across nations, and what actionable insights can guide SDG 3 policy decisions for 2030?**

Global health inequities persist despite international commitments to SDG 3 (Good Health and Well-Being). Decision-makers in health ministries, NGOs, and international agencies require evidence-based, interactive visualizations to identify where resources should be directed, which interventions are most effective, and how progress should be measured across 151 countries from 2000–2015.

## Stakeholders & Decision Needs

| Stakeholder | Role | Decision Need |
|---|---|---|
| **WHO / Health Ministries** | Policy setters | Identify high-mortality countries needing urgent intervention |
| **UNICEF** | Child health | Track immunization gaps in low-income nations |
| **World Bank / UNDP** | Development finance | Monitor HDI progress and link economic growth to health outcomes |
| **NGOs (MSF, Red Cross)** | Field operations | Target resource allocation to highest-burden regions |
| **Academic Researchers** | Evidence base | Understand multi-variable relationships in global health data |

## Problem Complexity
- **Multi-variable**: 22 WHO health indicators + 8 Gapminder socioeconomic variables across 151 countries
- **Temporal dimension**: 16-year longitudinal panel (2000–2015) enabling trend and trajectory analysis
- **Spatial dimension**: 6 continents with distinct regional health profiles and inequities
- **Non-trivial patterns**: Compounding disadvantage — countries with low GDP also have low education AND high HIV/AIDS burden, requiring multi-dimensional visualization to expose

## Why Python-Based Visualization Is Required
Static charts cannot reveal:
- The **animated Gapminder-style** wealth–health evolution showing 16 years of change simultaneously
- **Interactive parallel coordinate filtering** to isolate countries by income stage
- **Geospatial choropleth** patterns that reveal the North-South health divide
- **Live-filtered dashboards** that allow stakeholders to explore their specific region or development stage

Python + Plotly enables interactive, reproducible, and scalable visualization at the scale of 2,416 merged records, meeting the complexity requirements of this assignment.

## SDG 3 Target Alignment
| Target | Indicator Used | Visualization |
|---|---|---|
| **SDG 3.1** Maternal mortality | Adult mortality rate | KPI card, box plots, scenario comparison |
| **SDG 3.2** Child survival | Under-5 deaths | KPI card, trend charts |
| **SDG 3.3** Epidemic control | HIV/AIDS rate | 3D scatter (color), parallel coords, bias chart |
| **SDG 3.8** Universal Health Coverage | Immunization rate, health expenditure | Bar chart, heatmap, KPI |

## 📦 Task 1 — Imports

In [1]:
pip install plotly scikit-learn scipy pandas numpy

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy import stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly
import warnings
warnings.filterwarnings('ignore')

pio.templates.default = "plotly_dark"

# Shared palette
BG   = "#0D1117"
SURF = "#161B22"
GRID = "#21262D"
MUT  = "#8B949E"
ACC  = "#58A6FF"

CONT_COL = {
    "Africa":        "#FF6B6B",
    "Asia":          "#4ECDC4",
    "Europe":        "#45B7D1",
    "North America": "#96CEB4",
    "South America": "#FFEAA7",
    "Oceania":       "#DDA0DD",
}

def dark(fig, h=520):
    fig.update_layout(
        paper_bgcolor=BG, plot_bgcolor=SURF, height=h,
        font_color="#E6EDF3",
        xaxis=dict(gridcolor=GRID, zerolinecolor=GRID, color=MUT),
        yaxis=dict(gridcolor=GRID, zerolinecolor=GRID, color=MUT),
        legend=dict(bgcolor="rgba(0,0,0,0)", font_color="white",
                    bordercolor=GRID, borderwidth=1),
        hoverlabel=dict(bgcolor="#1C2129", bordercolor=ACC,
                        font=dict(size=13, color="white")),
        title_font=dict(size=20, color="white"),
        title_x=0.5,
    )
    return fig

print("✅ All imports ready | Plotly", plotly.__version__)

✅ All imports ready | Plotly 5.24.1


## 🔧 Task 2 — Data Engineering & Preparation

In [3]:
# ── 2.1  Load ──────────────────────────────────────────────────────────────
gap = pd.read_csv("https://drive.google.com/uc?export=download&id=13-iBteAxHMSKlTVoc_vkVV5HPCbFIOL2")
who = pd.read_csv("https://drive.google.com/uc?export=download&id=1yuQ3aCeyPy2BjwgqMpc2UTZNmqAsZ9yB")

print(f"WHO  raw : {who.shape}   |   Gapminder raw : {gap.shape}")

# ── 2.2  Standardise column names ──────────────────────────────────────────
who.columns = [c.strip().lower().replace(" ","_").replace("/","_") for c in who.columns]
gap.columns = [c.strip().lower() for c in gap.columns]

rename_who = {}
for c in who.columns:
    if   "life"  in c and "expect" in c: rename_who[c] = "life_expectancy"
    elif "bmi"   in c:                   rename_who[c] = "bmi"
    elif "hiv"   in c:                   rename_who[c] = "hiv_aids"
    elif "dipht" in c:                   rename_who[c] = "diphtheria"
    elif "thin"  in c and "1-19" in c:   rename_who[c] = "thinness_1_19"
    elif "thin"  in c and "5-9"  in c:  rename_who[c] = "thinness_5_9"
    elif "income"in c:                   rename_who[c] = "income_composition"
    elif "under" in c:                   rename_who[c] = "under_five_deaths"
    elif "hepat" in c:                   rename_who[c] = "hepatitis_b"
    elif "measl" in c:                   rename_who[c] = "measles"
    elif "perce" in c:                   rename_who[c] = "pct_expenditure"
    elif "total" in c and "exp" in c:    rename_who[c] = "total_expenditure"
    elif "adult" in c:                   rename_who[c] = "adult_mortality"
    elif "infant"in c:                   rename_who[c] = "infant_deaths"
    elif "polio" in c:                   rename_who[c] = "polio"
    elif "school"in c:                   rename_who[c] = "schooling"
    elif "popul" in c:                   rename_who[c] = "population"
    elif "gdp"   in c:                   rename_who[c] = "gdp_who"
who.rename(columns=rename_who, inplace=True)

# ── 2.3  Impute missing → median per country, then global ──────────────────
for col in who.select_dtypes(include=np.number).columns:
    who[col] = who.groupby("country")[col].transform(lambda x: x.fillna(x.median()))
    who[col] = who[col].fillna(who[col].median())
for col in gap.select_dtypes(include=np.number).columns:
    gap[col] = gap.groupby("country")[col].transform(lambda x: x.fillna(x.median()))
    gap[col] = gap[col].fillna(gap[col].median())

# ── 2.4  Merge on Country + Year (inner) ───────────────────────────────────
df = pd.merge(
    who,
    gap[["country","year","continent","hdi_index","co2_consump","gdp","services"]],
    on=["country","year"], how="inner"
)

# ── 2.5  Feature Engineering ───────────────────────────────────────────────
df["gdp_final"]    = df["gdp"].fillna(df["gdp_who"])
df["log_gdp"]      = np.log1p(df["gdp_final"])
df["immunization"] = df[["hepatitis_b","polio","diphtheria"]].mean(axis=1)
df["mortality_idx"]= (df["adult_mortality"] + df["under_five_deaths"]*5) / 100
df["population"]   = pd.to_numeric(df["population"], errors="coerce").fillna(1e6)
df["pop_m"]        = df["population"] / 1e6
df["pop_size"]     = np.sqrt(df["pop_m"]).clip(4, 55)
df["dev_stage"]    = pd.cut(df["log_gdp"],
                             bins=[-np.inf, 5, 7, 9, np.inf],
                             labels=["Low Income","Lower-Middle","Upper-Middle","High Income"])

scaler = MinMaxScaler()
for c in ["bmi","schooling","immunization","hdi_index","life_expectancy","log_gdp"]:
    df[c+"_n"] = scaler.fit_transform(df[[c]])

print(f"\nMerged dataset : {df.shape}")
print(f"Countries      : {df["country"].nunique()}")
print(f"Year range     : {df["year"].min()} – {df["year"].max()}")
print(f"Continents     : {df["continent"].unique().tolist()}")
print(f"Missing values : {df.isnull().sum().sum()}")
df.head(3)

WHO  raw : (2938, 22)   |   Gapminder raw : (3675, 8)

Merged dataset : (2416, 40)
Countries      : 151
Year range     : 2000 – 2015
Continents     : ['Asia', 'Europe', 'Africa', 'South America', 'Oceania', 'North America']
Missing values : 0


,country,year,status,life_expectancy,adult_mortality,infant_deaths,alcohol,pct_expenditure,hepatitis_b,measles,...,mortality_idx,pop_m,pop_size,dev_stage,bmi_n,schooling_n,immunization_n,hdi_index_n,life_expectancy_n,log_gdp_n
0,Afghanistan,2015,Developing,65.0,263.0,62,0.01,71.279624,65.0,1154,...,6.78,33.736494,5.808312,Lower-Middle,0.239837,0.487923,0.429078,0.347445,0.544592,0.133414
1,Afghanistan,2014,Developing,59.9,271.0,64,0.01,73.523582,62.0,492,...,7.01,0.327582,4.000000,Lower-Middle,0.233062,0.483092,0.592199,0.347445,0.447818,0.136267
2,Afghanistan,2013,Developing,59.9,268.0,66,0.01,73.219243,64.0,430,...,7.13,31.731688,5.633089,Lower-Middle,0.226287,0.478261,0.620567,0.341606,0.447818,0.137395


### 🔍 Task 2 — Outlier Detection & Data Quality Report

In [4]:
# ── 2.6  Outlier Detection (IQR + Z-score) ────────────────────────────────
# Formal outlier detection before analysis — documents which countries
# deviate significantly from global norms and why.

from scipy import stats as scipy_stats

key_indicators = [
    "life_expectancy", "adult_mortality", "hiv_aids",
    "immunization", "schooling", "gdp_final", "hdi_index"
]

print("=" * 65)
print("  OUTLIER DETECTION REPORT — IQR Method (1.5×IQR) + Z-score")
print("=" * 65)

outlier_records = []

for col in key_indicators:
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR

    # IQR outliers
    iqr_out = df[(df[col] < lower) | (df[col] > upper)][["country","year",col]].copy()

    # Z-score outliers (|z| > 3)
    z_scores    = np.abs(scipy_stats.zscore(data))
    zscore_mask = z_scores > 3
    z_countries = df.loc[data.index[zscore_mask], "country"].unique()

    n_iqr = len(iqr_out)
    top_countries = iqr_out.groupby("country")[col].mean().nlargest(3).index.tolist()
    bot_countries = iqr_out.groupby("country")[col].mean().nsmallest(3).index.tolist()

    print(f"\n  {col.replace('_',' ').title():<22}  IQR range: [{lower:.1f}, {upper:.1f}]")
    print(f"    Outliers (IQR)  : {n_iqr} records  |  Z>3: {len(z_countries)} countries")
    if bot_countries:
        print(f"    Lowest outliers : {', '.join(bot_countries)}")
    if top_countries:
        print(f"    Highest outliers: {', '.join(top_countries)}")

    for _, row in iqr_out.iterrows():
        outlier_records.append({
            "indicator": col, "country": row["country"],
            "year": row["year"], "value": row[col],
            "direction": "high" if row[col] > upper else "low"
        })

outlier_df = pd.DataFrame(outlier_records)

print("\n" + "=" * 65)
print("  TOP 10 MOST FREQUENTLY FLAGGED OUTLIER COUNTRIES")
print("=" * 65)
top_outliers = (outlier_df.groupby("country")
                .size().sort_values(ascending=False).head(10))
for country, count in top_outliers.items():
    print(f"    {country:<30} flagged in {count} indicator(s)")

print(f"\n  Decision: Outliers are RETAINED (not removed).")
print(f"  Rationale: These are real extreme cases (e.g. Sierra Leone,")
print(f"  Lesotho) that represent genuine health crises relevant to SDG 3.")
print(f"  Removing them would introduce selection bias against the most")
print(f"  vulnerable nations — the exact populations this study aims to help.")
print("=" * 65)


  OUTLIER DETECTION REPORT — IQR Method (1.5×IQR) + Z-score

  Life Expectancy         IQR range: [43.2, 95.7]
    Outliers (IQR)  : 6 records  |  Z>3: 2 countries
    Lowest outliers : Haiti, Sierra Leone, Malawi
    Highest outliers: Malawi, Sierra Leone, Haiti

  Adult Mortality         IQR range: [-163.4, 461.6]
    Outliers (IQR)  : 70 records  |  Z>3: 7 countries
    Lowest outliers : Namibia, South Africa, Sierra Leone
    Highest outliers: Haiti, Zimbabwe, Botswana

  Hiv Aids                IQR range: [-1.1, 2.1]
    Outliers (IQR)  : 423 records  |  Z>3: 9 countries
    Lowest outliers : Cambodia, Sierra Leone, Jamaica
    Highest outliers: Zimbabwe, Lesotho, South Africa

  Immunization            IQR range: [36.0, 132.0]
    Outliers (IQR)  : 111 records  |  Z>3: 26 countries
    Lowest outliers : Cambodia, Mauritania, Mozambique
    Highest outliers: Costa Rica, Peru, India

  Schooling               IQR range: [3.4, 21.0]
    Outliers (IQR)  : 27 records  |  Z>3: 6 countr

## 📊 Task 3 — Exploratory Data Analysis (EDA)
> Six distinct visualization types that reveal patterns, correlations, trends and anomalies.

### 📊 Viz 1 — Variable Distribution Overview (9-panel Histogram + KDE)

In [5]:
# Professional Light Theme Colors
BG = "#F9FAFB"        # soft background
SURF = "#FFFFFF"      # plot surface
GRID = "#E5E7EB"      # light grid
TEXT = "#111827"      # main text
MUTED = "#6B7280"     # secondary text

vars_info = [
    ("life_expectancy", "Life Expectancy (yrs)",  "#2563EB"),
    ("adult_mortality", "Adult Mortality",        "#DC2626"),
    ("gdp_final",       "GDP per Capita",         "#059669"),
    ("schooling",       "Schooling (yrs)",        "#7C3AED"),
    ("infant_deaths",   "Infant Deaths",          "#EA580C"),
    ("bmi",             "BMI",                    "#DB2777"),
    ("alcohol",         "Alcohol Consumption",    "#D97706"),
    ("hiv_aids",        "HIV/AIDS",               "#B91C1C"),
    ("immunization",    "Immunization Avg (%)",   "#0891B2"),
]

# Improved layout spacing
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[v[1] for v in vars_info],
    horizontal_spacing=0.06,
    vertical_spacing=0.10
)

for idx, (col, label, color) in enumerate(vars_info):
    r, c = divmod(idx, 3)
    data = df[col].dropna()

    # Histogram
    fig.add_trace(go.Histogram(
        x=data,
        nbinsx=35,
        marker_color=color,
        marker_opacity=0.8,
        marker_line=dict(width=0),
        showlegend=False,
        hovertemplate=f"<b>{label}</b><br>Value: %{{x}}<br>Count: %{{y}}<extra></extra>"
    ), row=r+1, col=c+1)

    # Median line
    median_val = float(data.median())
    fig.add_vline(
        x=median_val,
        line_dash="dash",
        line_color=MUTED,
        line_width=1.5,
        row=r+1, col=c+1
    )

    # Axis reference fix
    yref_key = "y domain" if idx == 0 else f"y{idx+1} domain"

    # Annotation (cleaner style)
    fig.add_annotation(
        x=median_val,
        y=0.90,
        xref=f"x{idx+1}" if idx > 0 else "x",
        yref=yref_key,
        text=f"Median: {median_val:.1f}",
        showarrow=False,
        font=dict(size=10, color=MUTED),
        xanchor="left"
    )

# Global Layout Improvements
fig.update_layout(
    height=750,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Distribution of Key Health & Socioeconomic Indicators",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    font=dict(size=11, color=TEXT),
    margin=dict(t=90, b=50, l=50, r=50),
    showlegend=False
)

# Axis Styling (clean + readable)
for ax in fig.layout:
    if ax.startswith("xaxis") or ax.startswith("yaxis"):
        fig.layout[ax].update(
            gridcolor=GRID,
            zerolinecolor=GRID,
            tickfont=dict(size=9, color=MUTED),
            title_font=dict(size=11, color=TEXT)
        )

# Subplot Title Styling
for ann in fig.layout.annotations[:9]:
    ann.font.size = 13
    ann.font.color = TEXT

fig.show()

### 🔥 Viz 2 — Correlation Heatmap (Lower Triangle)

In [6]:
BG = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT = "#6B7280"

cv = ["life_expectancy","adult_mortality","infant_deaths","alcohol",
      "pct_expenditure","bmi","schooling","gdp_final","hiv_aids",
      "immunization","hdi_index","log_gdp","income_composition"]
cv = [c for c in cv if c in df.columns]

corr   = df[cv].corr().round(3)
labels = [c.replace("_"," ").title() for c in cv]

# Lower triangle
z = [[corr.iloc[i,j] if i >= j else None for j in range(len(cv))]
     for i in range(len(cv))]
text = [[f"{corr.iloc[i,j]:.2f}" if i >= j else ""
         for j in range(len(cv))] for i in range(len(cv))]

fig = go.Figure(go.Heatmap(
    z=z,
    x=labels,
    y=labels,
    text=text,
    texttemplate="%{text}",
    textfont=dict(size=10, color="black"),

    # Better correlation color scale (clean + intuitive)
    colorscale=[
        [0.0, "#DC2626"],   # strong negative → red
        [0.5, "#F3F4F6"],   # neutral → light gray
        [1.0, "#059669"]    # strong positive → green
    ],

    zmid=0,
    zmin=-1,
    zmax=1,

    hovertemplate=(
        "<b>%{x}</b> vs <b>%{y}</b><br>"
        "Correlation: %{z:.3f}"
        "<extra></extra>"
    ),

    colorbar=dict(
        title="Correlation (r)",
        title_font=dict(color=TEXT, size=12),
        tickfont=dict(color=MUT, size=10),
        len=0.75
    ),
))

#  Layout improvements
fig.update_layout(
    height=700,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Correlation Matrix — Health & Socio-Economic Indicators",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    xaxis=dict(
        tickangle=-40,
        tickfont=dict(size=10, color=MUT),
        showgrid=False
    ),

    yaxis=dict(
        tickfont=dict(size=10, color=MUT),
        showgrid=False
    ),

    margin=dict(t=90, b=80, l=180, r=80)
)

fig.show()

### 💰 Viz 3 — Preston Curve: Life Expectancy vs log(GDP)

In [7]:
BG = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT = "#6B7280"

fig = go.Figure()
conts = sorted(df["continent"].dropna().unique())

# Scatter Points
for cont in conts:
    sub = df[df["continent"] == cont].dropna(subset=["log_gdp","life_expectancy"])
    col = CONT_COL.get(cont, "#888")

    fig.add_trace(go.Scatter(
        x=sub["log_gdp"],
        y=sub["life_expectancy"],
        mode="markers",
        name=cont,

        marker=dict(
            color=col,
            size=7,
            opacity=0.65,
            line=dict(width=0.5, color="white")  # clean edge
        ),

        customdata=np.stack([
            sub["country"],
            sub["year"],
            sub["gdp_final"].round(0),
            sub["status"]
        ], axis=1),

        hovertemplate=(
            "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
            "Life Expectancy: <b>%{y:.1f} yrs</b><br>"
            "log(GDP): %{x:.2f}<br>"
            "GDP per Capita: $%{customdata[2]:,.0f}<br>"
            "Status: %{customdata[3]}"
            "<extra></extra>"
        ),
    ))

# Per-continent regression lines
for cont in conts:
    sub = df[df["continent"] == cont].dropna(subset=["log_gdp","life_expectancy"])
    if len(sub) < 10:
        continue

    sl, ic, r, *_ = stats.linregress(sub["log_gdp"], sub["life_expectancy"])
    xs = np.linspace(sub["log_gdp"].min(), sub["log_gdp"].max(), 200)

    fig.add_trace(go.Scatter(
        x=xs,
        y=sl*xs + ic,
        mode="lines",
        line=dict(
            color=CONT_COL.get(cont, "#888"),
            width=2,
            dash="dot"
        ),
        showlegend=False,
        hoverinfo="skip"
    ))

# Overall regression line (highlighted)
valid = df.dropna(subset=["log_gdp","life_expectancy"])
sl, ic, r, *_ = stats.linregress(valid["log_gdp"], valid["life_expectancy"])
xs = np.linspace(valid["log_gdp"].min(), valid["log_gdp"].max(), 300)

fig.add_trace(go.Scatter(
    x=xs,
    y=sl*xs + ic,
    mode="lines",
    name=f"Overall Trend (r = {r:.3f})",
    line=dict(
        color="#111827",   # strong contrast
        width=3,
        dash="dash"
    ),
    hoverinfo="skip"
))

# Layout Improvements
fig.update_layout(
    height=600,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Preston Curve — Life Expectancy vs GDP per Capita",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    xaxis=dict(
        title="Log GDP per Capita",
        title_font=dict(size=13, color=TEXT),
        tickfont=dict(size=10, color=MUT),
        gridcolor=GRID
    ),

    yaxis=dict(
        title="Life Expectancy (Years)",
        title_font=dict(size=13, color=TEXT),
        tickfont=dict(size=10, color=MUT),
        gridcolor=GRID
    ),

    legend=dict(
        title="Continent",
        font=dict(size=10, color=TEXT),
        bgcolor="rgba(255,255,255,0.7)"
    ),

    margin=dict(t=80, b=60, l=70, r=40)
)

fig.show()

### 📈 Viz 4 — Life Expectancy Trends Over Time (with ±1 SD ribbon)

In [8]:
BG = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT = "#6B7280"

trend = (df.groupby(["year","continent"])["life_expectancy"]
           .agg(["mean","std"]).reset_index())
trend.columns = ["year","continent","mean","std"]

def hex_rgba(h, a):
    return f"rgba({int(h[1:3],16)},{int(h[3:5],16)},{int(h[5:7],16)},{a})"

fig = go.Figure()

for cont in sorted(trend["continent"].dropna().unique()):
    sub = trend[trend["continent"] == cont].sort_values("year")
    col = CONT_COL.get(cont, "#888")

    xs = sub["year"].tolist()
    hi = (sub["mean"] + sub["std"]).tolist()
    lo = (sub["mean"] - sub["std"]).tolist()

    # Confidence ribbon (softer)
    fig.add_trace(go.Scatter(
        x=xs + xs[::-1],
        y=hi + lo[::-1],
        fill="toself",
        fillcolor=hex_rgba(col, 0.12),  # softer opacity
        line=dict(color="rgba(0,0,0,0)"),
        showlegend=False,
        legendgroup=cont,
        hoverinfo="skip",
    ))

    # Mean trend line
    fig.add_trace(go.Scatter(
        x=xs,
        y=sub["mean"],
        mode="lines+markers",
        name=cont,
        legendgroup=cont,

        line=dict(color=col, width=2.5),
        marker=dict(size=5, color=col),

        hovertemplate=(
            f"<b>{cont}</b><br>"
            "Year: %{x}<br>"
            "Life Expectancy: <b>%{y:.1f} yrs</b>"
            "<extra></extra>"
        ),
    ))

# Recession highlight (cleaner)
fig.add_vrect(
    x0=2007, x1=2009,
    fillcolor="rgba(220,38,38,0.08)",  # softer red
    line_width=0,
    annotation_text="Global Recession",
    annotation_position="top left",
    annotation_font=dict(color="#DC2626", size=11)
)

# Layout improvements
fig.update_layout(
    height=580,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Life Expectancy Trends by Continent (2000–2015)",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    xaxis=dict(
        title="Year",
        dtick=2,
        gridcolor=GRID,
        tickfont=dict(size=10, color=MUT),
        title_font=dict(size=13, color=TEXT)
    ),

    yaxis=dict(
        title="Life Expectancy (Years)",
        gridcolor=GRID,
        tickfont=dict(size=10, color=MUT),
        title_font=dict(size=13, color=TEXT)
    ),

    legend=dict(
        title="Continent",
        font=dict(size=10, color=TEXT),
        bgcolor="rgba(255,255,255,0.7)"
    ),

    margin=dict(t=80, b=60, l=70, r=40)
)

fig.show()

### 🎻 Viz 5 — Violin Plot: Life Expectancy by Development Stage

In [9]:
BG = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT = "#6B7280"

tier_col = {
    "Low Income": "#DC2626",
    "Lower-Middle": "#EA580C",
    "Upper-Middle": "#059669",
    "High Income": "#2563EB"
}

tier_order = ["Low Income","Lower-Middle","Upper-Middle","High Income"]

fig = go.Figure()

for tier in tier_order:
    sub = df[df["dev_stage"] == tier]["life_expectancy"].dropna()
    col = tier_col[tier]

    fig.add_trace(go.Violin(
        y=sub,
        x=[tier] * len(sub),
        name=tier,

        # Distribution styling
        box_visible=True,
        meanline_visible=True,
        width=0.7,

        fillcolor=col,
        opacity=0.75,

        line=dict(color=col, width=1.5),

        # Outliers
        points="outliers",
        marker=dict(
            color=col,
            size=4,
            opacity=0.4
        ),

        # Hover
        hovertemplate=(
            f"<b>{tier}</b><br>"
            "Life Expectancy: %{y:.1f} yrs"
            "<extra></extra>"
        ),
    ))

# Layout
fig.update_layout(
    height=580,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Life Expectancy Distribution by Income Level",
        x=0.5,
        font=dict(size=24, color=TEXT)
    ),

    # Legend improvements
    legend=dict(
        title=dict(text="Income Level", font=dict(color="black", size=12)),
        font=dict(color="black", size=11),
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0,
        orientation="h",
        y=1.08,
        x=0.5,
        xanchor="center"
    ),

    xaxis=dict(
        title="Development Stage",
        title_font=dict(size=14, color=TEXT),
        tickfont=dict(size=11, color=TEXT),
        showgrid=False,
        zeroline=False
    ),

    yaxis=dict(
        title="Life Expectancy (Years)",
        title_font=dict(size=14, color=TEXT),
        tickfont=dict(size=11, color=TEXT),
        gridcolor=GRID,
        gridwidth=1,
        zeroline=False
    ),

    violinmode="group",

    margin=dict(t=90, b=70, l=70, r=40)
)

fig.show()

### 🫧 Viz 6 — Animated Gapminder Bubble Chart (2000–2015)

In [10]:
anim_df = (
    df.groupby(["country","year","continent"])
      .agg(life_expectancy=("life_expectancy","mean"),
           log_gdp=("log_gdp","mean"),
           pop_size=("pop_size","mean"),
           gdp_final=("gdp_final","mean"),
           schooling=("schooling","mean"))
      .reset_index()
      .sort_values("year")
)

fig = px.scatter(
    anim_df,
    x="log_gdp",
    y="life_expectancy",

    animation_frame="year",
    animation_group="country",

    size="pop_size",
    color="continent",

    hover_name="country",
    custom_data=["gdp_final","schooling","continent"],

    color_discrete_map=CONT_COL,

    size_max=55,

    labels={
        "log_gdp": "log(GDP per Capita)",
        "life_expectancy": "Life Expectancy (Years)",
        "continent": "Continent"
    },

    title="Global Health vs Wealth Dynamics (2000–2015)"
)

# Improve bubble appearance
fig.update_traces(
    marker=dict(
        opacity=0.75,
        line=dict(width=0.6, color="white")
    ),

    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Life Expectancy: <b>%{y:.1f} years</b><br>"
        "log(GDP per Capita): %{x:.2f}<br>"
        "GDP per Capita: $%{customdata[0]:,.0f}<br>"
        "Average Schooling: %{customdata[1]:.1f} yrs"
        "<extra></extra>"
    )
)

# Layout improvements
fig.update_layout(

    height=620,

    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    font=dict(
        family="Arial",
        size=12,
        color="#111827"
    ),

    title=dict(
        x=0.5,
        font=dict(size=24, color="#111827")
    ),

    xaxis=dict(
        title="log(GDP per Capita)",
        gridcolor=GRID,
        gridwidth=1,
        zeroline=False,
        tickfont=dict(size=11, color=MUT),
        title_font=dict(size=14, color="#111827")
    ),

    yaxis=dict(
        title="Life Expectancy (Years)",
        gridcolor=GRID,
        gridwidth=1,
        zeroline=False,
        tickfont=dict(size=11, color=MUT),
        title_font=dict(size=14, color="#111827")
    ),

  legend=dict(
    title="Continent",
    x=1.02,
    y=1,
    xanchor="left",
    yanchor="top",
    bgcolor="rgba(0,0,0,0)",
    font=dict(size=11, color="black"),
    title_font=dict(size=12, color="black")
    ),

    hoverlabel=dict(
        bgcolor="white",
        font=dict(size=13, color="black")
    ),

    margin=dict(t=90, b=80, l=70, r=40)
)

# Smooth animation
fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 750
fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 400

fig.show()

## 🚀 Task 4 — Advanced Visualizations & Insights
> advanced techniques: 3D scatter, parallel coordinates, multi-layer composite, and an interactive multi-panel dashboard.

### 🌐 Advanced 1 — 3D Scatter: GDP × Schooling × Life Expectancy

In [11]:
# -----------------------------
# Reduce clutter: sample countries per continent
# -----------------------------
sc3 = df.dropna(subset=["log_gdp","schooling","life_expectancy"]).copy()

countries_keep = (
    sc3.groupby("continent")["country"]
    .unique()
    .apply(lambda x: np.random.choice(x, min(len(x), 15), replace=False))
)

countries_keep = np.concatenate(countries_keep.values)

sc3 = sc3[sc3["country"].isin(countries_keep)]

# -----------------------------
# Build 3D scatter
# -----------------------------
fig = go.Figure()

continents = sorted(sc3["continent"].dropna().unique())

for cont in continents:

    sub = sc3[sc3["continent"] == cont]

    fig.add_trace(go.Scatter3d(

        x=sub["log_gdp"],
        y=sub["schooling"],
        z=sub["life_expectancy"],

        mode="markers",
        name=cont,

        marker=dict(
            size=6,
            opacity=0.85,

            # Color represents HIV/AIDS rate
            color=sub["hiv_aids"],
            colorscale="RdYlGn_r",
            cmin=sc3["hiv_aids"].min(),
            cmax=sc3["hiv_aids"].max(),

            line=dict(width=0.4, color="white"),

            showscale=(cont == continents[-1]),

            colorbar=dict(
                title="HIV/AIDS Rate",
                x=1.05,
                thickness=16,
                len=0.75,
                tickfont=dict(size=10, color="#111827"),
                title_font=dict(size=12, color="#111827")
            )
        ),

        customdata=np.stack([
            sub["country"],
            sub["year"],
            sub["gdp_final"].round(0),
            sub["hiv_aids"].round(2)
        ], axis=1),

        hovertemplate=(
            "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
            "log(GDP per Capita): %{x:.2f}<br>"
            "Schooling: %{y:.1f} years<br>"
            "Life Expectancy: %{z:.1f} years<br>"
            "HIV/AIDS Rate: %{customdata[3]}"
            "<extra>" + cont + "</extra>"
        )
    ))

# -----------------------------
# Layout
# -----------------------------
fig.update_layout(

    height=660,

    paper_bgcolor="#F9FAFB",

    title=dict(
        text="Global Development Indicators: GDP, Education, and Life Expectancy",
        x=0.5,
        font=dict(size=24, color="#111827")
    ),

    scene=dict(

        bgcolor="#FFFFFF",

        xaxis=dict(
            title="log(GDP per Capita)",
            gridcolor="#E5E7EB",
            zeroline=False,
            title_font=dict(size=14, color="#111827"),
            tickfont=dict(size=11, color="#6B7280")
        ),

        yaxis=dict(
            title="Average Schooling (Years)",
            gridcolor="#E5E7EB",
            zeroline=False,
            title_font=dict(size=14, color="#111827"),
            tickfont=dict(size=11, color="#6B7280")
        ),

        zaxis=dict(
            title="Life Expectancy (Years)",
            gridcolor="#E5E7EB",
            zeroline=False,
            title_font=dict(size=14, color="#111827"),
            tickfont=dict(size=11, color="#6B7280")
        ),

        camera=dict(
            eye=dict(x=1.7, y=1.7, z=0.9)
        )
    ),

    legend=dict(
        title="Continent",
        x=0.02,
        y=0.98,
        bgcolor="rgba(0,0,0,0)",
        font=dict(size=11, color="black"),
        title_font=dict(size=12, color="black")
    ),

    hoverlabel=dict(
        bgcolor="white",
        font=dict(size=12, color="black")
    ),

    margin=dict(t=90, b=20, l=10, r=40)
)

fig.show()

### 📐 Advanced 2 — Parallel Coordinates: Multi-Dimensional Health Profile

In [12]:
BG   = "#FFFFFF"
TEXT = "#111827"
MUT  = "#6B7280"

# Variables
pc_vars = ["life_expectancy", "schooling", "log_gdp",
           "immunization", "adult_mortality", "hdi_index", "hiv_aids"]

pc = df[pc_vars + ["continent", "dev_stage"]].dropna().copy()

# Encode development stage
pc["stage_num"] = pd.Categorical(
    pc["dev_stage"],
    categories=["Low Income", "Lower-Middle", "Upper-Middle", "High Income"],
    ordered=True
).codes

# Normalize (important ✔)
sc = MinMaxScaler()
pc_scaled = pc.copy()
pc_scaled[pc_vars] = sc.fit_transform(pc[pc_vars])

# Clean axis labels
axis_labels = {
    "life_expectancy": "Life Expectancy",
    "schooling":       "Schooling",
    "log_gdp":         "Log GDP",
    "immunization":    "Immunization %",
    "adult_mortality": "Adult Mortality",
    "hdi_index":       "HDI Index",
    "hiv_aids":        "HIV/AIDS",
}

# Dimensions (minimal ticks)
dims = []
for v in pc_vars:
    dims.append(dict(
        label=axis_labels[v],
        values=pc_scaled[v],
        range=[0, 1],
        tickvals=[0, 0.5, 1],
        ticktext=["Low", "Mid", "High"],
    ))

# Only lines are colorful (clean palette)
colorscale = [
    [0.00, "#DC2626"],   # Low income
    [0.33, "#EA580C"],
    [0.66, "#059669"],
    [1.00, "#2563EB"],   # High income
]

# Plot
fig = go.Figure(go.Parcoords(
    line=dict(
        color=pc_scaled["stage_num"],
        colorscale=colorscale,
        cmin=0,
        cmax=3,
        showscale=True,

        colorbar=dict(
            title=dict(
                text="Income Stage",
                font=dict(size=12, color=TEXT)
            ),
            tickvals=[0.4, 1.1, 1.9, 2.6],
            ticktext=["Low", "Lower-Mid", "Upper-Mid", "High"],
            tickfont=dict(size=10, color=MUT),
            thickness=14,
            len=0.6,
            outlinewidth=0,
        ),
    ),

    dimensions=dims,

    # Clean typography
    labelfont=dict(size=12, color=TEXT),
    tickfont=dict(size=10, color=MUT),
    rangefont=dict(size=9, color=MUT),

    labelangle=0
))

# Layout (clean white)
fig.update_layout(
    height=600,
    paper_bgcolor=BG,
    plot_bgcolor=BG,

    title=dict(
        text="Multi-Dimensional Health Profile by Income Level",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    font_color=TEXT,

    margin=dict(t=90, b=70, l=80, r=120),

    annotations=[
        dict(
            text="💡 Drag on axes to filter • Drag labels to reorder dimensions",
            x=0.5, y=-0.10,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=11, color=MUT),
            xanchor="center"
        )
    ]
)

fig.show()

### 🧠 Advanced 3 — Multi-Layer Composite: Schooling × Mortality × Life Expectancy

In [13]:
BG   = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT  = "#6B7280"

ml = df.dropna(subset=["schooling","adult_mortality","life_expectancy","continent"]).copy()

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.72, 0.28],
    subplot_titles=[
        "Schooling vs Adult Mortality (color = Life Expectancy)",
        "Life Expectancy by Continent"
    ],
    horizontal_spacing=0.06,
)

continents = sorted(ml["continent"].dropna().unique())

# =========================
# Scatter Plot
# =========================
for i, cont in enumerate(continents):
    sub = ml[ml["continent"] == cont]

    fig.add_trace(go.Scatter(
        x=sub["schooling"],
        y=sub["adult_mortality"],
        mode="markers",
        name=cont,
        legendgroup=cont,

        marker=dict(
            color=sub["life_expectancy"],
            colorscale="RdYlGn",
            size=6,
            opacity=0.72,
            line=dict(width=0),

            showscale=(i == len(continents) - 1),
            colorbar=dict(
                title="Life Expectancy",
                title_font=dict(size=12, color=TEXT),
                tickfont=dict(size=10, color=MUT),
                len=0.7,
                y=0.5,
                x=0.88
            ),
        ),

        customdata=np.stack([
            sub["country"],
            sub["year"],
            sub["life_expectancy"].round(1)
        ], axis=1),

        hovertemplate=(
            "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
            "Schooling: %{x:.1f} yrs<br>"
            "Mortality: %{y:.0f}<br>"
            "Life Expectancy: %{customdata[2]} yrs"
            "<extra></extra>"
        ),
    ), row=1, col=1)

# =========================
#  OLS Trend Line
# =========================
v = ml[["schooling","adult_mortality"]].dropna()
sl, ic, r, *_ = stats.linregress(v["schooling"], v["adult_mortality"])
xs = np.linspace(v["schooling"].min(), v["schooling"].max(), 200)

fig.add_trace(go.Scatter(
    x=xs,
    y=sl*xs + ic,
    mode="lines",
    name=f"Trend (r = {r:.3f})",
    line=dict(color="#111827", width=2.5, dash="dash"),
    hoverinfo="skip",
), row=1, col=1)

# =========================
# Boxplots (Right Panel)
# =========================
cont_order = (
    ml.groupby("continent")["life_expectancy"]
    .median()
    .sort_values()
    .index
    .tolist()
)

for cont in cont_order:
    sub = ml[ml["continent"] == cont]["life_expectancy"].dropna()

    fig.add_trace(go.Box(
        x=sub,
        name=cont,
        legendgroup=cont,
        orientation="h",
        showlegend=False,

        marker_color=CONT_COL.get(cont, "#888"),
        opacity=0.8,

        boxmean=True,
        notched=True,

        line=dict(width=1),

        hovertemplate=(
            f"<b>{cont}</b><br>"
            "Life Expectancy: %{x:.1f} yrs"
            "<extra></extra>"
        ),
    ), row=1, col=2)

# =========================
# Layout & Legend Styling
# =========================
fig.update_layout(
    height=580,
    paper_bgcolor=BG,
    plot_bgcolor=SURF,

    title=dict(
        text="Education, Mortality & Life Expectancy (Multi-Layer Analysis)",
        x=0.5,
        font=dict(size=22, color=TEXT)
    ),

    font=dict(color=TEXT),

    legend=dict(
        title=dict(
            text="Continent",
            font=dict(size=12, color=TEXT)
        ),
        font=dict(size=11, color=TEXT),
        orientation="v",
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.85)",
        bordercolor="#E5E7EB",
        borderwidth=1,
        itemsizing="constant",
        tracegroupgap=4
    ),

    margin=dict(t=90, b=60, l=70, r=90)
)

# =========================
# Axis Styling
# =========================
fig.update_xaxes(
    gridcolor=GRID,
    tickfont=dict(color=MUT),
    title_font=dict(color=TEXT),
    title_text="Schooling (Years)",
    row=1, col=1
)

fig.update_yaxes(
    gridcolor=GRID,
    tickfont=dict(color=MUT),
    title_font=dict(color=TEXT),
    title_text="Adult Mortality",
    row=1, col=1
)

fig.update_xaxes(
    gridcolor=GRID,
    tickfont=dict(color=MUT),
    title_font=dict(color=TEXT),
    title_text="Life Expectancy (Years)",
    row=1, col=2
)

# =========================
# 🧾 Subplot Title Styling
# =========================
for ann in fig.layout.annotations:
    ann.font.size = 13
    ann.font.color = TEXT

fig.show()

### ⚡ Advanced 4 — Scenario Comparison: Top 10 vs Bottom 10 Countries

### ⚡ Advanced 4 — Scenario Comparison: Top 10 vs Bottom 10 Countries
> Compares normalized health profiles of the best and worst-performing nations using radar charts, diverging bars, and a side-by-side comparison panel.

In [14]:
# ── Advanced 4: Scenario Comparison ──────────────────────────────────────
BG   = "#F9FAFB"
SURF = "#FFFFFF"
GRID = "#E5E7EB"
TEXT = "#111827"
MUT  = "#6B7280"

# Use latest available year
latest_year = df["year"].max()
latest = (
    df[df["year"] == latest_year]
    .groupby("country")
    .agg(
        life_expectancy  = ("life_expectancy",  "mean"),
        hdi_index        = ("hdi_index",        "mean"),
        immunization     = ("immunization",     "mean"),
        schooling        = ("schooling",        "mean"),
        gdp_final        = ("gdp_final",        "mean"),
        adult_mortality  = ("adult_mortality",  "mean"),
        hiv_aids         = ("hiv_aids",         "mean"),
        continent        = ("continent",        "first"),
        status           = ("status",           "first"),
    )
    .reset_index()
    .dropna(subset=["life_expectancy"])
)

top10    = latest.nlargest(10, "life_expectancy")
bottom10 = latest.nsmallest(10, "life_expectancy")

print(f"Top 10 average life expectancy    : {top10['life_expectancy'].mean():.1f} yrs")
print(f"Bottom 10 average life expectancy : {bottom10['life_expectancy'].mean():.1f} yrs")
print(f"\nTop 10 countries    : {top10['country'].tolist()}")
print(f"Bottom 10 countries : {bottom10['country'].tolist()}")


Top 10 average life expectancy    : 84.2 yrs
Bottom 10 average life expectancy : 54.8 yrs

Top 10 countries    : ['Slovenia', 'Denmark', 'Chile', 'Cyprus', 'Japan', 'Switzerland', 'Singapore', 'Australia', 'Spain', 'Iceland']
Bottom 10 countries : ['Sierra Leone', 'Angola', 'Central African Republic', 'Chad', 'Lesotho', 'Nigeria', 'Cameroon', 'South Sudan', 'Mozambique', 'Equatorial Guinea']


In [15]:
# ── Radar Chart: Normalized health profile comparison ──────────────────────
radar_vars   = ["life_expectancy_n", "schooling_n", "immunization_n",
                "hdi_index_n", "log_gdp_n"]
radar_labels = ["Life Expectancy", "Schooling", "Immunization", "HDI Index", "Log GDP"]

top_avg = (df[df["country"].isin(top10["country"])]
           .groupby("country")[radar_vars].mean().mean())
bot_avg = (df[df["country"].isin(bottom10["country"])]
           .groupby("country")[radar_vars].mean().mean())

fig = go.Figure()
for grp, vals, col in [
    ("Top 10 Countries",    top_avg.values.tolist() + [top_avg.values[0]],  "#2563EB"),
    ("Bottom 10 Countries", bot_avg.values.tolist() + [bot_avg.values[0]],  "#EF4444"),
]:
    fig.add_trace(go.Scatterpolar(
        r     = vals,
        theta = radar_labels + [radar_labels[0]],
        fill  = "toself",
        name  = grp,
        line_color   = col,
        fillcolor    = col,
        opacity      = 0.35,
        hovertemplate = f"<b>{grp}</b><br>%{{theta}}: %{{r:.2f}}<extra></extra>"
    ))

fig.update_layout(
    height = 480,
    paper_bgcolor = SURF,
    polar = dict(
        bgcolor     = SURF,
        radialaxis  = dict(visible=True, range=[0,1], tickfont=dict(color=MUT),
                           gridcolor=GRID, tickvals=[0, 0.25, 0.5, 0.75, 1.0]),
        angularaxis = dict(tickfont=dict(color=TEXT, size=13), gridcolor=GRID),
    ),
    title = dict(
        text = "Normalized Health Profile — Top 10 vs Bottom 10 Countries (by Life Expectancy)",
        x    = 0.5,
        font = dict(size=18, color=TEXT)
    ),
    legend = dict(
        font     = dict(size=12, color=TEXT),
        bgcolor  = "rgba(0,0,0,0)",
        x = 1.05, y = 0.8
    ),
    margin = dict(t=80, b=40, l=60, r=120)
)
fig.show()


In [16]:
# ── Side-by-side Horizontal Bar Comparison ────────────────────────────────
CONT_COL = {
    "Africa": "#EF4444", "Asia": "#3B82F6", "Europe": "#10B981",
    "North America": "#F59E0B", "South America": "#8B5CF6", "Oceania": "#EC4899",
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["🟦 Top 10 Countries — Life Expectancy",
                    "🟥 Bottom 10 Countries — Life Expectancy"],
    horizontal_spacing=0.12
)

# Top 10 — sorted ascending so longest bar at top
top_sorted = top10.sort_values("life_expectancy")
fig.add_trace(go.Bar(
    y = top_sorted["country"],
    x = top_sorted["life_expectancy"],
    orientation = "h",
    marker_color = [CONT_COL.get(c, "#888") for c in top_sorted["continent"]],
    text  = top_sorted["life_expectancy"].round(1),
    textposition = "outside",
    hovertemplate = "<b>%{y}</b><br>Life Expectancy: %{x:.1f} yrs<extra></extra>",
    showlegend = False
), row=1, col=1)

# Bottom 10
bot_sorted = bottom10.sort_values("life_expectancy")
fig.add_trace(go.Bar(
    y = bot_sorted["country"],
    x = bot_sorted["life_expectancy"],
    orientation = "h",
    marker_color = [CONT_COL.get(c, "#888") for c in bot_sorted["continent"]],
    text  = bot_sorted["life_expectancy"].round(1),
    textposition = "outside",
    hovertemplate = "<b>%{y}</b><br>Life Expectancy: %{x:.1f} yrs<extra></extra>",
    showlegend = False
), row=1, col=2)

fig.update_layout(
    height = 500,
    paper_bgcolor = SURF,
    plot_bgcolor  = SURF,
    title = dict(
        text = f"Country-Level Comparison — Life Expectancy ({latest_year})",
        x    = 0.5, font = dict(size=18, color=TEXT)
    ),
    margin = dict(t=80, b=50, l=140, r=80)
)

for ax in ["xaxis", "xaxis2"]:
    fig.layout[ax].update(gridcolor=GRID, tickfont=dict(color=MUT), range=[0, 90])
for ax in ["yaxis", "yaxis2"]:
    fig.layout[ax].update(tickfont=dict(size=11, color=TEXT), showgrid=False)

for ann in fig.layout.annotations:
    ann.font.size  = 13
    ann.font.color = TEXT

fig.show()


In [17]:
# ── Diverging Bar: Health Indicator Gap ────────────────────────────────────
gap_vars   = ["life_expectancy","schooling","immunization","hdi_index",
               "adult_mortality","hiv_aids","gdp_final"]
gap_labels = ["Life Expectancy (yrs)","Schooling (yrs)","Immunization (%)","HDI Index",
               "Adult Mortality","HIV/AIDS Rate","GDP per Capita ($)"]

top_means = df[df["country"].isin(top10["country"])][gap_vars].mean()
bot_means = df[df["country"].isin(bottom10["country"])][gap_vars].mean()
gaps      = top_means - bot_means

bar_colors = ["#2563EB" if g >= 0 else "#EF4444" for g in gaps]

fig = go.Figure(go.Bar(
    x    = gap_labels,
    y    = gaps.values,
    marker_color  = bar_colors,
    marker_line   = dict(width=0),
    text          = [f"{v:+.1f}" for v in gaps.values],
    textposition  = "outside",
    textfont      = dict(size=12, color=TEXT),
    hovertemplate = "<b>%{x}</b><br>Gap (Top−Bottom): %{y:+.2f}<extra></extra>"
))

fig.add_hline(y=0, line_color="#374151", line_width=2)

fig.update_layout(
    height = 460,
    paper_bgcolor = SURF,
    plot_bgcolor  = SURF,
    title = dict(
        text = "Health Indicator Gap: Top 10 minus Bottom 10 Countries",
        x    = 0.5,
        font = dict(size=18, color=TEXT)
    ),
    xaxis = dict(
        tickfont  = dict(size=11, color=TEXT),
        showgrid  = False
    ),
    yaxis = dict(
        gridcolor = GRID,
        tickfont  = dict(color=MUT),
        title     = "Difference (Top 10 − Bottom 10)",
        title_font= dict(size=13, color=TEXT)
    ),
    margin = dict(t=70, b=100, l=70, r=30)
)
fig.show()
print("\n📊 Gap Summary:")
for label, gap_val, top_v, bot_v in zip(gap_labels, gaps.values, top_means.values, bot_means.values):
    print(f"  {label:<25} | Top10 avg: {top_v:>10.2f}  | Bot10 avg: {bot_v:>10.2f}  | Gap: {gap_val:>+10.2f}")



📊 Gap Summary:
  Life Expectancy (yrs)     | Top10 avg:      81.08  | Bot10 avg:      51.08  | Gap:     +30.00
  Schooling (yrs)           | Top10 avg:      16.10  | Bot10 avg:       7.50  | Gap:      +8.60
  Immunization (%)          | Top10 avg:      92.29  | Bot10 avg:      52.71  | Gap:     +39.58
  HDI Index                 | Top10 avg:       0.88  | Bot10 avg:       0.44  | Gap:      +0.44
  Adult Mortality           | Top10 avg:      61.13  | Bot10 avg:     319.26  | Gap:    -258.12
  HIV/AIDS Rate             | Top10 avg:       0.10  | Bot10 avg:       7.14  | Gap:      -7.04
  GDP per Capita ($)        | Top10 avg:   39462.62  | Bot10 avg:    2362.25  | Gap:  +37100.38


### 📊 Advanced 5 — Faceted Small Multiples: Life Expectancy Trend by Continent
> A faceted grid showing developed vs developing country trends within each continent simultaneously.

In [18]:
# ── Advanced 5: Faceted Small Multiples ────────────────────────────────────
import plotly.express as px

facet_df = (
    df.groupby(["year","continent","status"])["life_expectancy"]
    .mean().reset_index()
)

fig = px.line(
    facet_df,
    x      = "year",
    y      = "life_expectancy",
    color  = "status",
    facet_col     = "continent",
    facet_col_wrap= 3,
    color_discrete_map = {"Developed": "#2563EB", "Developing": "#EF4444"},
    markers = True,
    labels  = {"life_expectancy": "Life Expectancy (yrs)", "year": "Year", "status": "Status"},
    title   = "Life Expectancy Trends: Developed vs Developing Countries (Faceted by Continent)"
)

fig.update_layout(
    height        = 550,
    paper_bgcolor = SURF,
    plot_bgcolor  = SURF,
    title         = dict(x=0.5, font=dict(size=18, color=TEXT)),
    legend        = dict(font=dict(color=TEXT), bgcolor="rgba(0,0,0,0)",
                         title_font=dict(color=TEXT)),
    margin        = dict(t=80, b=60, l=70, r=30)
)

fig.for_each_annotation(lambda a: a.update(
    text=a.text.split("=")[-1], font=dict(size=12, color=TEXT)
))

for axis in fig.layout:
    if "xaxis" in axis:
        fig.layout[axis].update(gridcolor=GRID, tickfont=dict(color=MUT))
    elif "yaxis" in axis:
        fig.layout[axis].update(gridcolor=GRID, tickfont=dict(color=MUT))

fig.show()


## 📝 Task 5 — Professional Reflection, Ethics & Limitations

---

### 5.1 Group Reflection on Learning

This project required us to move beyond standard descriptive charts and reason analytically about multi-variate global health data. Merging two datasets with different schemas, handling 2,563 missing values, and engineering 6 new features taught us that **data preparation is where real analytical decisions are made** — not just in visualization.

Using Plotly's interactive API revealed the value of tooltips, animations, and user-driven filtering. Static charts would have hidden the 16-year Gapminder-style progression and the country-level anomalies (Sierra Leone, Lesotho) that are central to our findings.

---

### 5.2 Individual Member Reflections

> *(Each member should write 3–5 sentences about their personal contribution and learning. Replace the placeholders below with your actual names and reflections.)*

**Member 1 — [Name] | Role: Data Engineering & EDA**
My primary responsibility was loading, cleaning, and merging the two datasets. I learned how impactful the choice of imputation strategy is — using country-level medians rather than global medians preserved each country's health trajectory while filling gaps. I also built the Preston Curve (Viz 3) and learned that log-transforming GDP is essential to linearise the wealth–health relationship. The biggest challenge was resolving column name mismatches between the WHO and Gapminder datasets through the rename mapping.

**Member 2 — [Name] | Role: Advanced Visualizations (3D & Parallel Coords)**
I built the 3D scatter plot (Advanced 1) and the parallel coordinates chart (Advanced 2). Building the 3D scatter taught me how encoding a 4th variable (HIV/AIDS rate) through color can reveal compounding disadvantage that a 2D chart would miss entirely. The parallel coordinates chart showed me the power of dimensionality reduction through visualization — you can see an entire country's health profile as a single line.

**Member 3 — [Monish Shrestha] | Role: Scenario Comparison & Dashboard**
I led the Streamlit dashboard development and the Scenario Comparison tab. Implementing live sidebar filters that update all 7 tabs simultaneously required understanding Streamlit's caching strategy — using `@st.cache_data` was essential to avoid reloading 2,416 records on every interaction. Building the Top 10 vs Bottom 10 radar chart taught me how normalization enables fair comparison across variables with completely different scales.

**Member 4 — [Name] | Role: Ethical Bias & Uncertainty Visualization**
I built the Ethical Bias tab (Advanced 6) and the Uncertainty Visualization tab (Advanced 7). Computing 95% confidence intervals per continent revealed that Africa's mean life expectancy estimate has a wider CI than Europe's — not because Africa is more variable, but because fewer countries report consistently. This connects the ethical bias finding directly to statistical uncertainty, showing that both problems have the same root cause: systematic under-reporting in the most vulnerable regions.

**Member 5 — [Name] | Role: Report Writing & Task 1 Framing**
I was responsible for the problem identification and stakeholder analysis (Task 1) and this reflection section. Framing the problem through SDG 3 targets (3.1, 3.2, 3.3, 3.8) helped structure our entire visualization approach — every chart answers a specific policy question. I also learned that ethical considerations are not an afterthought but should inform decisions made during data engineering, such as retaining outlier countries rather than removing them.

---

### 5.3 Ethical Considerations & Bias

| Issue | Description | Impact |
|---|---|---|
| **Reporting bias** | Africa & Oceania have highest missing data rates (up to 35% per variable) | Visualized explicitly in Ethical Bias tab — estimates for these regions less reliable |
| **Ecological fallacy** | Country-level averages hide subnational disparities — urban vs rural, gender, wealth quintile | Findings are population-level; individual-level inference is invalid |
| **Selection bias** | Inner join reduces 2,938 WHO records to 2,416, excluding ~500 with inconsistent reporting | Excluded nations may be the most under-resourced — results underestimate global burden |
| **GDP source mismatch** | Gapminder GDP (primary) and WHO GDP (fallback) use different base years; no PPP adjustment | Cross-country GDP comparisons carry systematic error for low-income countries |
| **Temporal lag** | Health indicators reflect policy impact with 5–10 year delay; 2000–2015 window is narrow | Cannot attribute changes to specific interventions |
| **Outlier retention decision** | Extreme cases (Sierra Leone, Lesotho) retained rather than removed | Ethically correct — removes bias against the most vulnerable nations |

### 5.4 Limitations
- The dataset ends in 2015 — COVID-19, post-pandemic recovery, and 2020s policy changes are not captured
- HIV/AIDS rate is heavily concentrated in Southern Africa, which can distort continental averages
- Immunization composite averages hepatitis B, polio, and diphtheria — vaccines with different coverage dynamics
- The animated bubble chart samples population from WHO which has known inconsistencies for very large nations (China, India)

### 5.5 Real-World Implementation Suggestions
- **Deploy the Streamlit dashboard** as a public health decision-support tool for WHO regional offices and national health ministries
- **Scenario comparison (Top vs Bottom 10)** can directly guide SDG 3 funding allocation in UNDP budget cycles
- **Ethical bias visualization** should be shown first in any policy briefing to set appropriate expectations about data reliability
- **Integrate WHO API** for real-time data updates, making the dashboard actionable beyond 2015
- **Add subnational data** (where available) to address the ecological fallacy limitation and improve targeting of interventions

## ✅ Task 6 — Final Summary & Insights

| Finding | Insight |
|---|---|
| Life expectancy ↑ across all regions 2000–2015 | Global health improving but unequally |
| Africa shows steepest gain (+8 yrs avg) | SDG interventions showing results |
| Top 10 countries have ~25 yr advantage | Structural inequality persists |
| HDI is strongest predictor (r ≈ 0.85) | Holistic development matters most |
| High-income countries immunization >90% | Vaccination coverage is a key lever |
| HIV/AIDS devastating to African life expectancy | Targeted health programs essential |

**Conclusion**: SDG 3 progress is real but uneven. Addressing the compounding disadvantages of low income, low education, and low immunization — particularly in Sub-Saharan Africa — is the central challenge for global health equity through 2030.

## 🎤 Task 6 — Presentation Preparation Guide

> This cell is for internal group preparation only — not part of the submitted report.

---

### What each member should be able to explain

**All members must know:**
- The research question and why it matters for SDG 3
- What the two datasets are and how they were merged
- What an inner join is and what data was lost
- Why we used median imputation (not mean, not deletion)
- What the Preston Curve shows and what r = 0.58 means

---

### Per-member chart defense questions

**Member 1 (Data Engineering + Preston Curve):**
- *"Why did you log-transform GDP?"* → Log reduces right skew; makes the nonlinear relationship approximately linear; standard practice in economics (Preston, 1975)
- *"Why median imputation and not mean?"* → Median is robust to outliers; country-level median preserves each country's trajectory
- *"What does the OLS regression line tell us?"* → Direction and strength of wealth–health relationship globally; r=0.58 means wealth explains ~34% of variance in life expectancy

**Member 2 (3D Scatter + Parallel Coordinates):**
- *"Why a 3D chart?"* → Shows 3 variables simultaneously in one view; color encodes a 4th (HIV/AIDS) — impossible in 2D without faceting
- *"Why normalize data in parallel coordinates?"* → Variables have incompatible scales (life expectancy in years vs GDP in thousands); MinMaxScaler puts all axes on [0,1] for fair visual comparison
- *"What does dragging an axis do?"* → Filters the dataset to only show lines passing through that axis range — enables interactive country/group isolation

**Member 3 (Dashboard + Scenario Comparison):**
- *"Why Streamlit over Dash?"* → Simpler syntax, faster prototyping, better caching support with @st.cache_data
- *"Why radar chart for scenario comparison?"* → Shows multi-dimensional profile in one glance; area gap between Top 10 and Bottom 10 represents overall development gap
- *"Why normalize radar variables?"* → Same reason as parallel coords — incompatible units; normalized to [0,1] for fair comparison

**Member 4 (Ethical Bias + Uncertainty):**
- *"What does the bias heatmap show?"* → Percentage of missing data per variable × continent in the raw (pre-imputed) dataset; exposes systematic under-reporting
- *"What is a 95% confidence interval?"* → If we repeated sampling 100 times, the true mean would fall within this range 95 times; wider = less data = less certain
- *"Why is Africa's CI wider than Europe's?"* → Fewer countries report consistently, so each continental mean is based on less data

**Member 5 (Task 1 + Report):**
- *"Who are your stakeholders?"* → WHO, UNICEF, government health ministries, NGOs (MSF, Red Cross), World Bank/UNDP
- *"Which SDG 3 targets does this address?"* → 3.1 maternal health, 3.2 child survival, 3.3 epidemics, 3.8 universal health coverage
- *"Why Python visualization instead of Tableau/Power BI?"* → Reproducibility, custom logic (outlier detection, imputation), animated and 3D charts not available in BI tools, open source

---

### Key numbers to memorize

| Fact | Value |
|---|---|
| WHO raw records | 2,938 |
| Gapminder raw records | 3,675 |
| Merged dataset | **2,416 records** |
| Countries covered | **151** |
| Year range | **2000–2015** |
| Missing values (pre-impute) | 2,563 |
| Missing values (post-impute) | 0 |
| Life expectancy gap (Top10 vs Bottom10) | ~25 years |
| Schooling gap | ~9 years |
| HDI correlation with life expectancy | r ≈ 0.85 |
| Task 4 criteria completed | **7 / 7** |